# YieldMPNN 环境配置教程

## 概述

本 notebook 用于说明如何在当前项目中，为 `reaction_yield_nn`（YieldMPNN） 准备一个适合教学演示的现代环境，并验证源码路径、数据文件和关键依赖都可用。

这份材料对应整个教程系列的第 1 步，重点解决两个问题：

1. 原仓库代码位于项目根目录之外，如何仍然用“相对当前项目根目录”的方式稳定引用它。
2. 原仓库只写了非常简略的依赖说明，如何把它整理成 2026 年仍然容易安装和验证的教学环境。

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 定位项目根目录与关键路径 |
| 2 | 梳理原仓库结构与运行入口 |
| 3 | 创建教学环境 |
| 4 | 验证核心依赖 |
| 5 | 验证源码与数据文件 |
| 6 | 总结与下一步 |

## 1. 定位项目根目录与关键路径

### 为什么需要这一步？

YieldMPNN 的源码仓库和环境目录都不在当前项目根目录里面。如果我们直接把绝对路径写死，教程会很难迁移；如果只写 notebook 当前目录，相对路径又会在不同启动方式下失效。

因此本教程统一采用下面的策略：

- 先自动定位当前项目根目录 `Chemical_Synthesis`
- 再从该根目录出发，解析源码仓库、环境根目录和教学数据文件
- 所有 notebook 都复用同一套路径约定

### 源码对应

- 当前教程目录：`teaching_demos/5.reaction_yields_tutorial/YieldMPNN`
- 外部源码仓库：`reaction_yield_nn`
- 原仓库入口：`run_code.py`

### 下面这个代码单元会做什么？

它会统一解析 `PROJECT_ROOT / SOURCE_REPO / ENV_DIR / DEMO_DATA`，并打印存在性检查结果。换句话说，这一格的目标不是训练模型，而是先证明“教程引用的所有关键路径都能被稳定定位”。



In [1]:
from pathlib import Path
from pprint import pprint

# ====== Step 1: 定位项目根目录与关键路径 ======
def locate_project_root(start: Path | None = None, root_name: str = 'Chemical_Synthesis') -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if candidate.name == root_name:
            return candidate
    raise RuntimeError(f'Cannot locate project root named {root_name!r} from {path}')

PROJECT_ROOT = locate_project_root()
TUTORIAL_DIR = PROJECT_ROOT / 'teaching_demos/5.reaction_yields_tutorial/YieldMPNN'
SOURCE_REPO_REL = Path('../../../../../data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/reaction_yield_nn')
ENV_ROOT_REL = Path('../../../../../data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/envs')
SOURCE_REPO = (PROJECT_ROOT / SOURCE_REPO_REL).resolve()
ENV_ROOT = (PROJECT_ROOT / ENV_ROOT_REL).resolve()
ENV_DIR = ENV_ROOT / 'yieldmpnn_tutorial_envs'
ENV_YML = TUTORIAL_DIR / 'yieldmpnn_tutorial_env.yml'
DEMO_DATA = TUTORIAL_DIR / 'data/yieldmpnn_demo_reactions.csv'

path_summary = {
    'PROJECT_ROOT': PROJECT_ROOT,
    'TUTORIAL_DIR': TUTORIAL_DIR,
    'SOURCE_REPO_REL': SOURCE_REPO_REL,
    'SOURCE_REPO': SOURCE_REPO,
    'ENV_ROOT_REL': ENV_ROOT_REL,
    'ENV_ROOT': ENV_ROOT,
    'ENV_DIR': ENV_DIR,
    'ENV_YML': ENV_YML,
    'DEMO_DATA': DEMO_DATA,
}

pprint({key: str(value) for key, value in path_summary.items()})
print('\n存在性检查：')
for key in ['TUTORIAL_DIR', 'SOURCE_REPO', 'ENV_ROOT', 'ENV_YML', 'DEMO_DATA']:
    print(f'- {key}:', path_summary[key].exists())

{'DEMO_DATA': '/home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/5.reaction_yields_tutorial/YieldMPNN/data/yieldmpnn_demo_reactions.csv',
 'ENV_DIR': '/home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/yieldmpnn_tutorial_envs',
 'ENV_ROOT': '/home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs',
 'ENV_ROOT_REL': '../../../../../data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/envs',
 'ENV_YML': '/home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/5.reaction_yields_tutorial/YieldMPNN/yieldmpnn_tutorial_env.yml',
 'PROJECT_ROOT': '/home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis',
 'SOURCE_REPO': '/home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/reaction_yield_nn',
 'SOURCE_REPO_REL': '../../../../../data/ubuntu_work_beta/other_work/GNN_AIDD/Che

## 2. 原仓库结构与运行入口

### 为什么先看结构？

环境配置不是“把包装上就结束”。教学时更重要的是知道每个依赖服务于哪一段源码，否则后面遇到错误时很难判断是数据预处理、DGL 图构建，还是模型前向出问题。

### 核心模块

| 文件 | 作用 |
|------|------|
| `data/get_data.py` | 把原始 reaction SMILES 转成压缩图数组 `dataset_*.npz` |
| `dataset.py` | 从压缩数组恢复为 DGL 图对象 |
| `util.py` | batch 拼接与 MC dropout 辅助函数 |
| `model.py` | `MPNN`、`reactionMPNN`、训练与推理函数 |
| `run_code.py` | 原仓库主入口，串起数据加载、训练和评估 |

> 说明
>
> 原仓库 README 只给出了 `Python / PyTorch / DGL / RDKit` 这四个大类依赖。教学版需要把它们进一步细化成一个可验证的现代环境。

### 下面这个代码单元会做什么？

它会直接遍历外部源码仓库的目录树，让读者在真正安装环境之前，先建立“数据脚本、数据层、模型层、训练入口”之间的文件级映射。



In [2]:
# ====== Step 2: 浏览原仓库目录 ======
def preview_tree(root: Path, max_depth: int = 2):
    for path in sorted(root.rglob('*')):
        rel = path.relative_to(root)
        if len(rel.parts) <= max_depth:
            prefix = '  ' * (len(rel.parts) - 1)
            suffix = '/' if path.is_dir() else ''
            print(f'{prefix}{rel.name}{suffix}')

preview_tree(SOURCE_REPO, max_depth=2)

.git/
  HEAD
  branches/
  config
  description
  hooks/
  index
  info/
  logs/
  objects/
  packed-refs
  refs/
LICENSE
README.md
__pycache__/
  dataset.cpython-310.pyc
  model.cpython-310.pyc
  util.cpython-310.pyc
data/
  dataset_1_0.npz
  dataset_2_0.npz
  get_data.py
  split/
  test_1.npz
dataset.py
model.py
run_code.py
util.py


## 3. 创建教学环境

### 为什么这里不能简单写成“全都用最新版”？

截至 **2026-03-30**，PyTorch 官方已经提供更高版本的稳定包线；但 YieldMPNN 依赖 DGL，而 DGL 的官方 wheel 支持并不是对每个最新 PyTorch 版本都同步覆盖。

因此教学版采用的是“**截至当前日期，官方轮子可获得且兼容性更稳的现代组合**”：

- Python `3.10`
- PyTorch `2.6.0`
- torchvision `0.21.0`
- torchaudio `2.6.0`
- DGL `2.5.0`
- RDKit `2025.09.6`

这里的决策原则不是盲目追逐最新号，而是选择：

\[
\text{teaching environment} = \arg\max_{e} \; \text{reproducibility}(e) + \text{compatibility}(e) + \text{clarity}(e)
\]

也就是说，教学环境首先要满足三件事：

1. 能装上
2. 能运行原仓库的关键 API
3. 能稳定复现实验主线

### 源码对应

- 原仓库 README: 只列出依赖类别
- 教学版新增: `yieldmpnn_tutorial_env.yml`

### 与原仓库差异

教学版显式补上了：

- `numpy / pandas / scipy / scikit-learn`
- `jupyterlab / ipykernel`
- PyTorch CPU wheel 源
- DGL 官方 wheel 索引

### 下面这个代码单元会做什么？

它只负责打印教学环境配置文件本身。这个步骤的意图是先让读者看到“环境描述”，再执行真正的环境创建命令。



In [3]:
# ====== Step 3.1: 查看教学环境配置文件 ======
print(ENV_YML.read_text(encoding='utf-8'))

name: yieldmpnn_tutorial_envs
channels:
  - pytorch
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - numpy=2.2
  - pandas=2.3
  - scipy=1.15
  - scikit-learn=1.7
  - pytorch=2.5.1
  - torchvision=0.20.1
  - torchaudio=2.5.1
  - cpuonly
  - rdkit=2025.09.6
  - jupyterlab
  - ipykernel
  - matplotlib
  - seaborn
  - pip
  - pip:
      - dgl==2.4.0



### 这一步具体做什么？

下面这个代码单元会真正调用 `conda env create` 或 `conda env update`。它的意图是把上面的环境描述文件落成一个可执行环境，而不是只停留在静态配置。

输入：

- `yieldmpnn_tutorial_env.yml`
- 教程约定的环境路径 `ENV_DIR`

输出：

- 一个可执行的 `yieldmpnn_tutorial_envs` 环境

> 注意
>
> 如果某个依赖版本不可安装，这一格就会直接暴露问题，因此它也是整个教程里最重要的环境自检点。

In [5]:
%%bash
# ====== Step 3.2: 创建或更新教学环境 ======
set -euo pipefail

ROOT_DIR="$(python - <<'PY'
from pathlib import Path
path = Path.cwd().resolve()
for candidate in [path, *path.parents]:
    if candidate.name == 'Chemical_Synthesis':
        print(candidate)
        break
else:
    raise RuntimeError('Cannot locate Chemical_Synthesis root')
PY
)"

ENV_YML="$ROOT_DIR/teaching_demos/5.reaction_yields_tutorial/YieldMPNN/yieldmpnn_tutorial_env.yml"
ENV_DIR="$ROOT_DIR/../../../../../data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/envs/yieldmpnn_tutorial_envs"

mkdir -p "$(dirname "$ENV_DIR")"

if [ -x "$ENV_DIR/bin/python" ]; then
  conda env update -p "$ENV_DIR" -f "$ENV_YML" --prune
else
  conda env create -p "$ENV_DIR" -f "$ENV_YML"
fi

echo "Environment prepared at: $ENV_DIR"

Channels:
 - conda-forge
 - defaults
 - pytorch
Platform: linux-64

data.json): ...working... 

done
Solving environment: ...working... 

done




==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version:

...

_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/envs/yieldmpnn_tutorial_envs
#
# To deactivate an 

active environment, use
#
#     $ conda deactivate



Environment prepared at: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synt

hesis/../../../../../data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/envs/yieldmpnn_tut

orial_envs


### 这一步具体做什么？

下面的代码单元会把刚创建好的 conda 环境注册成一个独立的 Jupyter kernel。这样做的目的不是增加功能，而是减少后续 notebook 的运行歧义：

- 读者打开 `2_数据处理.ipynb` 和 `3_模型展示.ipynb` 时，可以直接切到 `YieldMPNN Tutorial`
- 代码会在同一个解释器、同一组依赖下执行
- 不会混用当前会话的临时内核和教学环境

In [6]:
%%bash
# ====== Step 3.3: 注册教学环境为 Jupyter Kernel ======
set -euo pipefail

ROOT_DIR="$(python - <<'PY'
from pathlib import Path
path = Path.cwd().resolve()
for candidate in [path, *path.parents]:
    if candidate.name == 'Chemical_Synthesis':
        print(candidate)
        break
else:
    raise RuntimeError('Cannot locate Chemical_Synthesis root')
PY
)"

ENV_DIR="$ROOT_DIR/../../../../../data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/envs/yieldmpnn_tutorial_envs"
"$ENV_DIR/bin/python" -m ipykernel install --user --name yieldmpnn_tutorial --display-name "YieldMPNN Tutorial"

echo "Kernel registered: YieldMPNN Tutorial"

Installed kernelspec yieldmpnn_tutorial in /tmp/codex_jupyter_8892_data/kernels/yieldmpnn_tutorial


Kernel registered: YieldMPNN Tutorial


## 4. 验证核心依赖

### 为什么这里用 `subprocess` 调目标环境的 Python？

因为读者可能在任意内核下打开这个 notebook。教学上更稳妥的做法，是**显式调用刚创建好的环境解释器**来检查版本，而不是假定当前 notebook 内核已经切换成功。

### 下面这个代码单元会做什么？

它不仅会 import 关键依赖并打印版本号，还会额外做一次很小的 `torch` 矩阵乘法，并检查 `NNConv / Set2Set` 是否可以被导入。这样可以把“只会 import”与“真正能做数值计算”区分开来。由于当前环境组合在 CPU 上需要稳定的线程层设置，这一格也会显式传入 `MKL_THREADING_LAYER=GNU` 与 `OMP_NUM_THREADS=1`。

In [9]:
import subprocess
import textwrap
import os

# ====== Step 4: 验证目标环境的核心依赖 ======
validation_script = textwrap.dedent(
    '''
    import sys
    import numpy as np
    import pandas as pd
    import torch
    import dgl
    import rdkit
    import scipy
    import sklearn
    from dgl.nn.pytorch import NNConv, Set2Set

    a = torch.randn(4, 4)
    b = torch.randn(4, 4)
    c = a @ b

    print('python', sys.version)
    print('numpy', np.__version__)
    print('pandas', pd.__version__)
    print('torch', torch.__version__)
    print('dgl', dgl.__version__)
    print('rdkit', rdkit.__version__)
    print('scipy', scipy.__version__)
    print('sklearn', sklearn.__version__)
    print('nnconv', NNConv.__name__)
    print('set2set', Set2Set.__name__)
    print('matmul shape', tuple(c.shape))
    '''
)

env = os.environ.copy()
env.setdefault('MKL_THREADING_LAYER', 'GNU')
env.setdefault('OMP_NUM_THREADS', '1')

result = subprocess.run(
    [str(ENV_DIR / 'bin/python'), '-c', validation_script],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Environment validation failed')

python 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]
numpy 2.2.6
pandas 2.3.3
torch 2.6.0+cpu
dgl 2.5.0
rdkit 2025.09.6
scipy 1.15.2
sklearn 1.7.2
nnconv NNConv
set2set Set2Set
matmul shape (4, 4)



## 5. 验证源码与数据文件

### 为什么这一节很重要？

YieldMPNN 的最小运行单元不是“只 import 一个模型类”，而是：

1. 读到原始 split 数据或预处理后的 `dataset_*.npz`
2. 能够从外部源码仓库导入 `dataset.py / model.py / util.py`
3. DGL 图对象和 RDKit 解析在同一个环境中都可用

这一步相当于为后面两份 notebook 做一个总联调。

### 下面这个代码单元会做什么？

它会在目标环境里读取原始 split 和预处理数据，并真正实例化一次 `GraphDataset`。同时它也会显式切到 `SOURCE_REPO` 目录再调用原仓库类，因为 `dataset.py` 内部把 `./data/...` 写成了相对当前工作目录的形式。

In [10]:
# ====== Step 5: 用目标环境验证源码导入与数据文件 ======
smoke_test_script = textwrap.dedent(
    f'''
    import sys
    from pathlib import Path
    import numpy as np

    source_repo = Path(r"{SOURCE_REPO}")
    sys.path.insert(0, str(source_repo))

    from dataset import GraphDataset

    raw_path = source_repo / 'data/split/data1_split_0.npz'
    processed_path = source_repo / 'data/dataset_1_0.npz'
    raw = np.load(raw_path, allow_pickle=True)['data_df']
    processed = np.load(processed_path, allow_pickle=True)

    import os
    old_cwd = os.getcwd()
    os.chdir(source_repo)
    try:
        data = GraphDataset(1, 0)
    finally:
        os.chdir(old_cwd)

    print('raw_path', raw_path)
    print('processed_path', processed_path)
    print('raw shape', raw.shape)
    print('processed keys', processed.files)
    print('dataset length', len(data))
    print('rmol_max_cnt', data.rmol_max_cnt)
    print('pmol_max_cnt', data.pmol_max_cnt)
    print('node_dim', data.rmol_node_attr[0].shape[1])
    print('edge_dim', data.rmol_edge_attr[0].shape[1])
    '''
)

env = os.environ.copy()
env.setdefault('MKL_THREADING_LAYER', 'GNU')
env.setdefault('OMP_NUM_THREADS', '1')

result = subprocess.run(
    [str(ENV_DIR / 'bin/python'), '-c', smoke_test_script],
    capture_output=True,
    text=True,
    cwd=str(SOURCE_REPO),
    env=env,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Smoke test failed')

raw_path /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/reaction_yield_nn/data/split/data1_split_0.npz
processed_path /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/reaction_yield_nn/data/dataset_1_0.npz
raw shape (3955, 2)
processed keys ['rmol', 'pmol', 'reaction']
dataset length 3955
rmol_max_cnt 6
pmol_max_cnt 1
node_dim 43
edge_dim 8



## 6. 总结

本 notebook 完成了三件事：

1. 把当前项目根目录、外部源码仓库和外部环境目录统一成一套相对路径约定。
2. 给出一份适合 2026 年教学复现的现代环境配置，而不是停留在原仓库 README 的粗粒度依赖描述。
3. 用目标环境解释器验证了核心依赖、源码导入和数据文件都可用。

### 下一步

- 下一份 notebook：`2_数据处理.ipynb`
- 重点内容：从 reaction SMILES 到 `dataset_*.npz` 的完整压缩流程